WRITTEN FOR COLAB 

Run these cells once to set up caiman and define the functions to be used below

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

#!pip install -Uqq ipdb
#import ipdb


# Install CaImAn - takes around 2 minutes
!rm -r '/content/CaImAn'
!git clone https://github.com/flatironinstitute/CaImAn.git
%cd '/content/CaImAn/'
!pip install -e .

# Install caiman dependencies (&> /dev/null will suppress the hundreds of printed lines in the output)
!pip install -r requirements.txt &> /dev/null

#import other dependencies
import cv2
import glob
import numpy as np
import os
import matplotlib.pyplot as plt
import imageio

#IMPORTANT! Newer versions of h5py will cause errors when saving results
!pip install h5py==2.10.0
import h5py

#IMPORTANT! Newer versions of ipyparallel will cause errors when saving starting a cluster
!pip install ipyparallel==8.1.0
import ipyparallel

#Set up caiman
!python setup.py build_ext -i

#Other file setup
#THIS NO LONGER APPEARS TO BE TRUE JUST INSTALL NEWER VERSION OF JINJA, LIKE 3.1.2
#IMPORTANT: NEW VERSION OF JINJA CAUSES ERROR, INSTALL THIS PREVIOUS VERSION BEFORE INSTALLING CAIMANMANAGER.PY
####!pip install Jinja2==2.10.1
!pip install Jinja2==3.1.2 #install this version now so it will run (earlier had to install old version so it would run)

!python caimanmanager.py install --inplace

#Caiman imports
import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf as cnmf
from caiman.source_extraction.cnmf import params as params
from caiman.utils.visualization import plot_contours, nb_view_patches, nb_plot_contour
#from caiman.summary_images import local_correlations_movie_offline
from scipy.ndimage import center_of_mass
from IPython.display import display, clear_output

In [ ]:
ipyparalell?

#logging and configuring enviroment for interactive visualizations
#Some of this is redundant

try:
    get_ipython().magic(u'load_ext autoreload')
    get_ipython().magic(u'autoreload 2')
    print(1)
except:
    print('NOT IPYTHON')

# from ipyparallel import Client
# import logging
# import matplotlib.pyplot as plt
# import numpy as np
# import os
# import psutil
# from scipy.ndimage.filters import gaussian_filter
# import sys

# import caiman as cm
# from caiman.utils.visualization import nb_view_patches3d
# import caiman.source_extraction.cnmf as cnmf
# from caiman.components_evaluation import evaluate_components, estimate_components_quality_auto
# from caiman.cluster import setup_cluster
# from caiman.paths import caiman_datadir

import bokeh.plotting as bpl
bpl.output_notebook()

#uncomment this to enable detailed logging for debugging

# logging.basicConfig(format=
#                           "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s",
#                     # filename="/tmp/caiman.log",
#                     level=logging.DEBUG)

Object `ipyparalell` not found.
1


In [ ]:
#create parameters object
opts = params.CNMFParams()
#fname will be assigned in the loop
fnames = []
subfolder = 'stacksfin' #'stackRaw_mc'
opts.motion['var_name_hdf5'] = subfolder
opts.data['var_name_hdf5'] = subfolder

# set initial values for extraction and evaluation
# most of these are specific to our data and will not need to be changed during optimization

# overall params about our data

fr = 20                 # approximate frame rate of data - CONFIRMED FPS
decay_time = .4         # length of transient - CONFIRMED APPROPRIATE FOR OUR INDICATOR GCaMP6f
dims = [128, 256]       # dimensions of the FOV in pixels - CONFIRMED
dxy = [.29, .29]        # resolution of 1 pixel in um, should be period - CONFIRMED BY CARL

opts.set('data', {'fnames': fnames,
                   'fr': fr,
                   'decay_time': decay_time,
                   #'dims': dims,
                   'dxy': dxy
                  })

# params related to the temporal traces

p = 0                   # order of the autoregressive system - 0 from carl's code
fudge_factor = 0.96        # (default is 0.96; Carl's value = 1) -- bias correction factor for discrete time constants
ITER = 2                # (default is 2; Carl's value=5) -- block coordinate descent iterations
nb = 4                  # temporal global background components - TUNE
bas_nonneg = True


opts.set('temporal', {'p': p,
                      'fudge_factor': fudge_factor,
                      'ITER': ITER,
                      'bas_nonneg': bas_nonneg,
                      'nb': nb
                 })

# p is also set in the preprocessing step
opts.set('preprocess', {'p': p
                 })



# params related to the FOV and patches for parallel processing

is_patches = False      # flag for processing in patches or not - turn on or off - Not used in Matlab

if is_patches:          # PROCESS IN PATCHES AND THEN COMBINE
    rf = 25             # half size of each patch
    stride = 5          # overlap between patches
    K = 3               # number of components in each patch
    p_patch = p

else:                   # PROCESS THE WHOLE FOV AT ONCE
    rf = None           # setting these parameters to None
    stride = None       # will run CNMF on the whole FOV
    K = 20              # number of neurons expected (in the whole FOV) - 40 from Carl's Code, seems to be too many

n_processes = 2         # Number of processes to run in parallel, 2 for 2 cores available in Colab

opts.set('patch', {'rf': rf,
                   'stride': stride,
                   'n_processes': n_processes,
                  })


# initialization params
ssub = 1               # spatial downsampling
tsub = 1                # temporal downsampling
ssub_B = 1              # background spatial downsampling
gSig = [3,3]            # radius (half-size) of average neurons (in pixels)
tau= (3,3)                   # standard deviation of neuron size along x and y - from Carl's code
method_init = 'greedy_roi' #python Caiman defaults to greedy_roi, carl's code uses sparse_nmf, but sparse_nmf runs MUCH slower
sigma_smooth_snmf = (.5,.5,.5)
perc_baseline_snmf = 50

opts.set('init', {'K': K,            # declared above in patch params
                   'tau': tau,
                   'tsub': tsub,
                   'ssub': ssub,
                   'ssub_B': ssub_B,
                   'gSig': gSig,
                   'nb': nb, #decleared in temporal
                   'method_init': method_init,
                   'sigma_smooth_snmf': sigma_smooth_snmf,
                   'perc_baseline_snmf': perc_baseline_snmf
                  })

# parameters related to merging correlated ROIs
merge_thr = 0.95     # merging threshold, max correlation allowed - From Carl's Code

opts.set('merging', {'thr': merge_thr
                            })


#set some spatial params
#snb = 1                  # spatial global background components
#update_background_components = False #use this??
opts.set('spatial', {'nb': nb #delcared above in init
                            })

# %% COMPONENT EVALUATION
# the components are evaluated in three ways:
#   a) the shape of each component must be correlated with the data
#   b) a minimum peak SNR is required over the length of a transient
#   c) each shape passes a CNN based classifier (this will pick up only neurons
#           and filter out active processes)


#Not sure if these should be tuned or not

#min_SNR = 0      # peak SNR for accepted components (if above this, acept)
SNR_lowest = 0         # minimum SNR for accepted components (if below this, reject)
rval_thr = 0.8     # space correlation threshold (if above this, accept)

use_cnn = False      # use the CNN classifier affects if 2 below params are used
min_cnn_thr = 0.9  # if cnn classifier predicts below this value, reject
cnn_lowest = 0.1   # neurons with cnn probability lower than this value are rejected

opts.set('quality', {#'min_SNR': min_SNR,
                     'SNR_lowest': SNR_lowest,
                     'rval_thr': rval_thr,
                     'use_cnn': use_cnn,
                     'min_cnn_thr': min_cnn_thr,
                     'cnn_lowest': cnn_lowest})


#Manually assign subfolder variable
opts.motion['var_name_hdf5'] = subfolder
opts.data['var_name_hdf5'] = subfolder


In [ ]:
#Function to run seeded cnmf using masks, then return results
def seeded_cnmf_detrended(path_to_stack, path_to_masks, opts):
  import warnings
  warnings.simplefilter(action='ignore', category=FutureWarning)

  fnames = [path_to_stack]
  opts.set('data', {'fnames': fnames})

  #cluster handling
  if 'dview' in locals():
    cm.stop_server(dview=dview)
  dview = cm.cluster.start_server(ncpus=2) #Start a cluster with 2 CPU's (available in colab)


  #Read in masks and reformat
  g = h5py.File(path_to_masks, 'r')

  #transpose the matrix and save to an array A
  mask_A = g['bwMaskStack'][:].T

  #close the .mat file holding the masks
  g.close()

  #rearrange the dimensions of masks to match reformatted .h5 movie that has been flipped (not necessary if input movie has not been flipped),
                                                                                                                #but will afect following lines
  mask_A = mask_A.transpose(1,0,2)

  #reshape to 2D, first dimension is 128*256 (32768), 2nd dimension is the # of ROI's
  mask_A = mask_A.reshape((mask_A.shape[1]*mask_A.shape[0]), mask_A.shape[2])

  #convert the values from 0/1 to boolean False/True
  mask_A = np.array(mask_A, dtype=bool)
  print('mask read in and reformatted')

  #For seeded CNMF, need to verify certain params
  opts.patch['only_init'] = False
  opts.data['use_cnn'] = False

  print('params adjusted for seeded cnmf')

  #%% MEMORY MAPPING
  # memory map the file in order 'C'
  fname_new = cm.save_memmap(fnames, base_name='memmap_', order='C', dview=dview)
  print(fname_new)

  # now load the file
  Yr, dims, T = cm.load_memmap(fname_new)
  images = np.reshape(Yr.T, [T] + list(dims), order='F')
  print(type(images))
  print(images.shape)

      #load frames in python format (T x X x Y)

  #Initialize a new cnmf object and pass in our masks as the "Ain" param
  #"Ain" is A-in, meaning the A matrix holding the spatial footprints of the roi's
  print('initializing cnmf object')
  cnm_seeded = cnmf.CNMF(n_processes = 2, params=opts, dview=dview, Ain=mask_A)
  print('cnmf object initialized')
  print('starting seeded cnmf')
  cnm_seeded.fit(images)
  print('seeded cnmf completed')
  print('starting component evaluation')
  cnm_seeded.estimates.evaluate_components(images, opts, dview=dview)
  print('component evaluation completed')
  print('detrending and normalizing temporal traces')
  cnm_seeded.estimates.detrend_df_f(detrend_only=False, flag_auto=False, quantileMin = 28, frames_window = 500)
  print('detrending and normalizing complete')

  cm.stop_server(dview=dview)

  print(path_to_stack[:(len(path_to_stack)-4)].split('/')[4])

  #return the cnm object and a dictionary of results
  return cnm_seeded, {'stack_name': path_to_stack[:(len(path_to_stack)-4)].split('/')[4], #chops the movie identifier out of the filepath
                        'spatial': np.transpose(cnm_seeded.estimates.A.A, axes = (1,0)), #de-sparsified (.A) and transposed to put the axes in roi x pixel order
                        'temporal': cnm_seeded.estimates.F_dff}


In [ ]:
#manually setting the paths to the example files in the new folder
#The whole path should be the same, only need to change the filenames at the end of the paths
!pip install Jinja2==3.1.2 #install this version now so it will run (earlier had to install old version so it would run)
#!pip install Jinja2==3.0.0 #install this version now so it will run (earlier had to install old version so it would run)

import datetime
import fnmatch
recordingIDs = ['22*_*']

for recordingID in recordingIDs:
  tmpdate = datetime.datetime.now().strftime("%Y%m%dT%H%M%S") #create extraction ID, one for each recordingID

  path_prefix_stack = '../drive/MyDrive/stacksfin/'
  path_prefix_mask = '../drive/MyDrive/masks/'
  mask_filename_template = recordingID + '_*_manualROIs.mat'

  path_to_masks = []

  for f in os.listdir(path_prefix_mask):
    if f.endswith('.mat'):
      if fnmatch.fnmatch(f,mask_filename_template):
        tmp = path_prefix_mask + f
        path_to_masks.append(tmp)


  for f2 in path_to_masks:

    path_to_stacks = []

    stackinsert = f2[:(len(f2)-4)].split('/')[4].split('_')
    stack_filename_template = stackinsert[0] + '_' + stackinsert[1] + '_*_stackRaw_mc_.mat'

    for f in os.listdir(path_prefix_stack):
      if f.endswith('.mat'):
        if fnmatch.fnmatch(f,stack_filename_template):
          tmp = path_prefix_stack + f
          path_to_stacks.append(tmp)

    for f in path_to_stacks:

      cnm_object, results = seeded_cnmf_detrended(path_to_stack = f, #from the pairs of files above, could be swapped out for string directly
                                        path_to_masks = f2, #same as above
                                        opts = opts) #params object globally initialized earlier

      #visualize results
      cnm_object.estimates.dims = [128,256]
      print(f"ROIs extracted from {results['stack_name']}")

      #procedurally generate the path to the result file we will write
      results_folder_path = '/content/drive/MyDrive/responses/'
      #save each stack's results to a different h5 file
      #generate filepath and filename for the results
      result_h5_path = results_folder_path + results['stack_name'] + 'result_' + tmpdate + '.h5'
      #Write the results into an hdf5 file
      with h5py.File(result_h5_path, "w") as result_file:
        result_file.create_dataset('stack_name', data=results['stack_name'])
        result_file.create_dataset('extractionID', data=tmpdate)
        result_file.create_dataset('temporal', data=results['temporal'])
        result_file.create_dataset('spatial', data=results['spatial'])

      cnm_object.estimates.nb_view_components(denoised_color = 'red')



Output hidden; open in https://colab.research.google.com to view.